# Group Classifier — Pollinator 4-Class

Trains a 4-class pollinator classifier on insect crops confirmed by `binary_classifier`.

**Classes:** `bumblebee` · `fly` · `butterfly
` · `other`

## Data sources
| Source | Purpose |
|---|---|
| `annotated_crops/labeled/{class}/` | Arctic field crops from `annotate.py` |
| `web_images/{class}/` | Web images (iNaturalist etc.) to supplement rare classes |

**Recommended minimum per class:** 200 crops (Arctic) + 200 web images for rare classes.
fly is the dominant class in Arctic data — do NOT oversample to 1:1, use weighted loss instead.

## Workflow
1. Run `ls_v4.ipynb` → crops
2. Run `annotate.py` → `annotated_crops/labeled/{bumblebee,fly,butterfly,other}/`
3. (Optional) Download web images → `web_images/{bumblebee,butterfly,other}/`
4. Run `binary_classifier.ipynb` first — group classifier only sees confirmed insect crops
5. Run this notebook → `models/group_best.pth`

## Backbone choice
EfficientNet-B2 (ImageNet pretrained, all layers trainable) is preferred over InsectNet here.
InsectNet was trained on North American photos, domain shift to noisy Arctic crops is severe.
EfficientNet-B2 with full fine-tune generalises better when data is limited.

## Two-stage training strategy
If web images are used:
- **Stage 1:** Train on web images + Arctic crops combined (backbone learns general insect features)
- **Stage 2:** Fine-tune on Arctic crops only with small lr (adapts to field conditions)
If only Arctic crops are available, skip Stage 1 and go straight to Stage 2.


In [ ]:
import json
import time
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as T
from PIL import Image
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler

try:
    from sklearn.metrics import classification_report, confusion_matrix
    HAS_SKLEARN = True
except ImportError:
    print('Install sklearn: pip install scikit-learn')
    HAS_SKLEARN = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'PyTorch: {torch.__version__}')


## Configuration

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import zipfile
with zipfile.ZipFile('/content/drive/MyDrive/pollinator-classification/Insects_images/annotated_crops.zip', 'r') as z:
    z.extractall('/content/')
with zipfile.ZipFile('/content/drive/MyDrive/pollinator-classification/Insects_images/web_images.zip', 'r') as z:
    z.extractall('/content/')
print("Done")

In [ ]:
# ── Data mode ──────────────────────────────────────────────────────────────
# 'ls'       → Lian's data only       (annotated_crops/labeled)
# 'mb'       → Marcus's data only     (annotated_crops_mb/labeled)
# 'combined' → both datasets merged
DATA_MODE = 'combined'   # 'ls', 'mb', or 'combined'

# ── Data paths ─────────────────────────────────────────────────────────────
_BASE = Path('/content')
_ARCTIC_LS = _BASE / 'labeled_ls'
_ARCTIC_MB = _BASE / 'labeled_mb'


if DATA_MODE == 'ls':
    ARCTIC_DIR = _ARCTIC_LS
elif DATA_MODE == 'mb':
    ARCTIC_DIR = _ARCTIC_MB
elif DATA_MODE == 'combined':
    ARCTIC_DIR = None   # handled in collect_samples below
else:
    raise ValueError(f'Unknown DATA_MODE: {DATA_MODE}')

# Web images downloaded from iNaturalist etc. to supplement rare classes.
# Set to None to skip web images entirely.
WEB_DIR = _BASE / 'web_images'   # or None
MODEL_OUT_DIR = Path(f'models_{DATA_MODE}')
MODEL_OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Classes ────────────────────────────────────────────────────────────────
# annotate.py uses 'butterfly', web images may also include moths.
# Both are merged into 'butterfly_moth' here.
# ARCTIC_ALIAS maps annotate.py folder names → canonical class names.
CLASSES = ['bumblebee', 'fly', 'butterfly_moth', 'other']
ARCTIC_ALIAS = {
    'bumblebee':  'bumblebee',
    'fly':        'fly',
    'butterfly':  'butterfly_moth',   # annotate.py calls it 'butterfly'
    'other':      'other',
    # 'unsure' and 'background' are intentionally excluded
}

# ── Model choice ───────────────────────────────────────────────────────────
# 'efficientnet' → EfficientNet-B2, ImageNet pretrained, all layers trainable
# 'insectnet'    → InsectNet backbone (RegNet-Y-32GF), frozen backbone
# 'both'         → train both and compare
MODEL = 'both'

INSECTNET_WEIGHTS = Path('/content/drive/MyDrive/pollinator-classification/InsectNet/model.pth')
UNFREEZE_LAST_BLOCK = True

EPOCHS_STAGE2_PER_MODEL = {
    'efficientnet': 15,
    'insectnet':     0,
}

# ── Training settings ──────────────────────────────────────────────────────
IMG_SIZE = 224
EPOCHS_STAGE1 = 20   # Stage 1: web + arctic combined (skip if WEB_DIR is None)
EPOCHS_STAGE2 = 15   # Stage 2: arctic only fine-tune
BATCH    = 32
LR_STAGE1 = 1e-3
LR_STAGE2 = 1e-4     # smaller lr for fine-tune stage
VAL_FRAC  = 0.2
SEED      = 42

# ── Verify data ────────────────────────────────────────────────────────────
_dirs_to_check = [_ARCTIC_LS, _ARCTIC_MB] if DATA_MODE == 'combined' else [ARCTIC_DIR]

print('Arctic crops:')
for _d in _dirs_to_check:
    print(f'\n{_d.name}:')
    for folder, cls in ARCTIC_ALIAS.items():
        d = _d / folder
        n = sum(len(list(d.glob(f'*.{e}'))) for e in ('jpg','jpeg','png')) if d.exists() else 0
        flag = '' if n >= 100 else '  ← low, consider web images'
        print(f'  {folder:15} → {cls:15}: {n:>5}{flag}')


## Dataset

In [ ]:
torch.manual_seed(SEED)
np.random.seed(SEED)

# ── Web images alias (global) ──────────────────────────────────────────────
WEB_ALIAS = {
    'bumblebee':      'bumblebee',
    'fly':            'fly',
    'butterfly':      'butterfly_moth',
    'butterfly_moth': 'butterfly_moth',
    'other':          'other',
}


def letterbox(img, size):
    """Pad image to square with black borders then resize. Preserves aspect ratio."""
    w, h = img.size
    max_side = max(w, h)
    sq = Image.new('RGB', (max_side, max_side), (0, 0, 0))
    sq.paste(img, ((max_side - w) // 2, (max_side - h) // 2))
    return sq.resize((size, size), Image.BILINEAR)


class CropDataset(Dataset):
    def __init__(self, samples, transform):
        self.samples   = samples
        self.transform = transform

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        return self.transform(img), label


def collect_arctic_samples(arctic_dir, arctic_alias, classes):
    """Collect Arctic crop samples from one labeled directory."""
    cls_to_idx    = {c: i for i, c in enumerate(classes)}
    arctic_samples = []

    if arctic_dir is None:
        return arctic_samples, {i: 0 for i in range(len(classes))}

    for folder, cls in arctic_alias.items():
        d = arctic_dir / folder
        if not d.exists():
            continue
        idx = cls_to_idx[cls]
        for ext in ('*.jpg', '*.jpeg', '*.png'):
            for p in d.glob(ext):
                arctic_samples.append((p, idx))

    counts = {i: 0 for i in range(len(classes))}
    for _, lbl in arctic_samples:
        counts[lbl] += 1

    print(f'\nArctic crops ({arctic_dir.name}):')
    for i, cls in enumerate(classes):
        print(f'  {cls:20}: {counts[i]:>5}')

    return arctic_samples, counts


def collect_web_samples(web_dir, classes):
    """Collect web image samples from web_images directory."""
    cls_to_idx  = {c: i for i, c in enumerate(classes)}
    web_samples = []
    counts_web  = {i: 0 for i in range(len(classes))}

    if not (web_dir and web_dir.exists()):
        return web_samples, counts_web

    for folder, cls in WEB_ALIAS.items():
        if cls not in cls_to_idx:
            continue
        d = web_dir / folder
        if not d.exists():
            continue
        idx = cls_to_idx[cls]
        for ext in ('*.jpg', '*.jpeg', '*.png'):
            for p in d.glob(ext):
                web_samples.append((p, idx))
                counts_web[idx] += 1

    print('\nWeb images per class:')
    for i, cls in enumerate(classes):
        print(f'  {cls:20}: {counts_web[i]:>5}')

    return web_samples, counts_web


def stratified_split(samples, classes, val_frac, seed, test_frac=0.1):
    """Stratified train/val/test split by class."""
    rng = np.random.default_rng(seed)
    by_class = {i: [] for i in range(len(classes))}
    for i, (_, lbl) in enumerate(samples):
        by_class[lbl].append(i)
    train_idx, val_idx, test_idx = [], [], []
    for lbl, idxs in by_class.items():
        idxs = list(idxs)
        rng.shuffle(idxs)
        n_test = max(1, int(len(idxs) * test_frac))
        n_val  = max(1, int(len(idxs) * val_frac))
        test_idx.extend(idxs[:n_test])
        val_idx.extend(idxs[n_test:n_test + n_val])
        train_idx.extend(idxs[n_test + n_val:])
    return train_idx, val_idx, test_idx


def make_transforms(img_size, augment=True):
    lb = T.Lambda(lambda img: letterbox(img, img_size))
    if augment:
        return T.Compose([
            lb,
            T.RandomHorizontalFlip(),
            T.RandomVerticalFlip(),
            T.RandomRotation(30),
            T.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.3, hue=0.05),
            T.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
            T.ToTensor(),
            T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])
    return T.Compose([
        lb,
        T.ToTensor(),
        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])


def make_loader(samples, indices, img_size, batch, augment=True, weighted_sampler=True):
    subset = [samples[i] for i in indices]
    tf = make_transforms(img_size, augment)
    ds = CropDataset(subset, tf)
    if weighted_sampler and augment:
        labels   = [s[1] for s in subset]
        cls_w    = 1.0 / np.maximum(np.bincount(labels, minlength=len(CLASSES)), 1)
        sample_w = [cls_w[l] for l in labels]
        sampler  = WeightedRandomSampler(sample_w, len(sample_w))
        return DataLoader(ds, batch_size=batch, sampler=sampler, num_workers=0)
    return DataLoader(ds, batch_size=batch, shuffle=augment, num_workers=0)


# ── Collect data ───────────────────────────────────────────────────────────
if DATA_MODE == 'combined':
    s1, ca1 = collect_arctic_samples(_ARCTIC_LS, ARCTIC_ALIAS, CLASSES)
    s2, ca2 = collect_arctic_samples(_ARCTIC_MB, ARCTIC_ALIAS, CLASSES)
    arctic_samples = s1 + s2
    counts_arctic  = {i: ca1[i] + ca2[i] for i in range(len(CLASSES))}
elif DATA_MODE == 'ls':
    arctic_samples, counts_arctic = collect_arctic_samples(_ARCTIC_LS, ARCTIC_ALIAS, CLASSES)
elif DATA_MODE == 'mb':
    arctic_samples, counts_arctic = collect_arctic_samples(_ARCTIC_MB, ARCTIC_ALIAS, CLASSES)
else:
    raise ValueError(f'Unknown DATA_MODE: {DATA_MODE}')

web_samples, counts_web = collect_web_samples(WEB_DIR, CLASSES)

# ── Split ──────────────────────────────────────────────────────────────────
# Mixed val: split on arctic + web combined for balanced per-class evaluation
all_samples_for_split = arctic_samples + web_samples
all_train_idx, all_val_idx, _ = stratified_split(all_samples_for_split, CLASSES, VAL_FRAC, SEED)  # test sets defined separately below

# Separate arctic and web indices
n_arctic = len(arctic_samples)
arctic_train_idx    = [i for i in all_train_idx if i < n_arctic]
web_train_idx       = [i - n_arctic for i in all_train_idx if i >= n_arctic]
arctic_val_idx      = all_val_idx   # mixed val: arctic + web

# Test Arctic: Arctic samples NOT in train or val
used_arctic  = set(arctic_train_idx) | {i for i in all_val_idx if i < n_arctic}
arctic_test_idx = [i for i in range(n_arctic) if i not in used_arctic]

# Test Web: web samples NOT in train or val, capped at same size as arctic test
used_web_train = set(web_train_idx)
used_web_val   = {i - n_arctic for i in all_val_idx if i >= n_arctic}
used_web       = used_web_train | used_web_val
web_test_pool  = [i for i in range(len(web_samples)) if i not in used_web]
# Stratified sample from web test pool to match arctic test size
rng_test = np.random.default_rng(SEED)
rng_test.shuffle(web_test_pool)
web_test_idx = web_test_pool[:len(arctic_test_idx)]

print(f'\nData split summary:')
print(f'  Train: {len(arctic_train_idx)} arctic + {len(web_train_idx)} web = {len(arctic_train_idx)+len(web_train_idx)} total')
print(f'  Val:   {len([i for i in arctic_val_idx if i < n_arctic])} arctic + {len([i for i in arctic_val_idx if i >= n_arctic])} web = {len(arctic_val_idx)} total')
print(f'  Test Arctic: {len(arctic_test_idx)} (Arctic only)')
print(f'  Test Web:    {len(web_test_idx)} (web only, capped to match Arctic test)')

print(f'\nArctic train per class:')
from collections import Counter
arctic_train_labels = [arctic_samples[i][1] for i in arctic_train_idx]
for i, cls in enumerate(CLASSES):
    print(f'  {cls:20}: {arctic_train_labels.count(i):>5}')

print(f'\nWeb train per class:')
web_train_labels = [web_samples[i][1] for i in web_train_idx]
for i, cls in enumerate(CLASSES):
    print(f'  {cls:20}: {web_train_labels.count(i):>5}')

## Model Builder

In [ ]:
def build_efficientnet_b2(num_classes=4):
    """EfficientNet-B2 pretrained on ImageNet. All layers trainable."""
    print(f'Building EfficientNet-B2 (ImageNet, {IMG_SIZE}px, all layers trainable)')
    model = torchvision.models.efficientnet_b2(weights='IMAGENET1K_V1')
    in_features = model.classifier[-1].in_features
    model.classifier[-1] = nn.Linear(in_features, num_classes)
    nn.init.xavier_uniform_(model.classifier[-1].weight)
    nn.init.zeros_(model.classifier[-1].bias)
    n = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'Trainable params: {n:,}')
    return model


def build_insectnet(weights_path, num_classes=4, unfreeze_last_block=UNFREEZE_LAST_BLOCK):
    """
    InsectNet backbone (RegNet-Y-32GF) with frozen backbone.
    Good when Arctic crop data is limited — avoids overfitting.
    Set unfreeze_last_block=True if val F1 is low after frozen training.
    """
    print(f'Building InsectNet group classifier (RegNet-Y-32GF, {IMG_SIZE}px)')
    if not weights_path.exists():
        raise FileNotFoundError(f'InsectNet weights not found: {weights_path}')
    model = torchvision.models.regnet_y_32gf()
    model.fc = nn.Linear(3712, 2526)
    state = torch.load(weights_path, map_location='cpu', weights_only=False)
    model.load_state_dict(state['model'] if 'model' in state else state, strict=True)
    print('InsectNet weights loaded ✓')
    model.fc = nn.Linear(3712, num_classes)
    nn.init.xavier_uniform_(model.fc.weight)
    nn.init.zeros_(model.fc.bias)
    for name, p in model.named_parameters():
        p.requires_grad = name.startswith('fc.')
    if unfreeze_last_block:
        print('Unfreezing last backbone block (trunk_output.block4) + fc')
        for name, p in model.named_parameters():
            if 'trunk_output.block4' in name or name.startswith('fc.'):
                p.requires_grad = True
    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_total = sum(p.numel() for p in model.parameters())
    print(f'Trainable: {n_train:,} / {n_total:,} params')
    return model


## Training Loop

In [ ]:
def weighted_criterion(counts, classes, device):
    """CrossEntropyLoss weighted inversely by class count."""
    w = torch.tensor(
        [1.0 / max(1, counts[i]) for i in range(len(classes))],
        dtype=torch.float, device=device
    )
    w = w / w.sum()
    return nn.CrossEntropyLoss(weight=w)


def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    loss_sum = correct = total = 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        out  = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        preds     = out.argmax(1)
        loss_sum += loss.item() * labels.size(0)
        correct  += (preds == labels).sum().item()
        total    += labels.size(0)
    return loss_sum / total, correct / total


@torch.no_grad()
def eval_epoch(model, loader, criterion, device, classes):
    model.eval()
    loss_sum = correct = total = 0
    all_p, all_l = [], []
    tp = {i: 0 for i in range(len(classes))}
    fp = {i: 0 for i in range(len(classes))}
    fn = {i: 0 for i in range(len(classes))}
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        out   = model(imgs)
        preds = out.argmax(1)
        loss_sum += criterion(out, labels).item() * labels.size(0)
        correct  += (preds == labels).sum().item()
        total    += labels.size(0)
        all_p.extend(preds.cpu().tolist())
        all_l.extend(labels.cpu().tolist())
        for i in range(len(classes)):
            tp[i] += ((preds == i) & (labels == i)).sum().item()
            fp[i] += ((preds == i) & (labels != i)).sum().item()
            fn[i] += ((preds != i) & (labels == i)).sum().item()
    per_class_f1 = {}
    for i in range(len(classes)):
        p = tp[i] / max(1, tp[i] + fp[i])
        r = tp[i] / max(1, tp[i] + fn[i])
        per_class_f1[classes[i]] = 2 * p * r / max(1e-8, p + r)
    macro_f1 = np.mean(list(per_class_f1.values()))
    return {
        'loss': loss_sum / total, 'acc': correct / total,
        'macro_f1': macro_f1, 'per_class_f1': per_class_f1,
        'preds': all_p, 'labels': all_l
    }


def run_stage(model, stage_name, train_loader, val_loader,
              epochs, lr, counts, ckpt_path, device):
    """One training stage. Saves best macro-F1 checkpoint."""
    criterion = weighted_criterion(counts, CLASSES, device)
    optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    best_f1 = 0.0
    history = {k: [] for k in ['tr_loss', 'val_loss', 'val_f1', 'val_acc']}

    print(f'\n{"="*70}')
    print(f'{stage_name}  |  img={IMG_SIZE}px  |  epochs={epochs}  |  lr={lr}')
    print(f'{"="*70}')
    print(f'{"Ep":>3}  {"TrLoss":>7}  {"VaLoss":>7}  {"MacroF1":>8}  {"Acc":>6}  '
          + '  '.join(f'{c[:4]:>6}' for c in CLASSES))

    t0 = time.time()
    for ep in range(1, epochs + 1):
        tr_loss, tr_acc = train_epoch(model, train_loader, optimizer, criterion, device)
        v = eval_epoch(model, val_loader, criterion, device, CLASSES)
        scheduler.step()

        history['tr_loss'].append(tr_loss)
        history['val_loss'].append(v['loss'])
        history['val_f1'].append(v['macro_f1'])
        history['val_acc'].append(v['acc'])

        star = ''
        if v['macro_f1'] > best_f1:
            best_f1 = v['macro_f1']
            torch.save({'epoch': ep, 'stage': stage_name,
                        'img_size': IMG_SIZE, 'classes': CLASSES,
                        'state_dict': model.state_dict(),
                        'val_macro_f1': v['macro_f1'],
                        'val_acc': v['acc']}, ckpt_path)
            star = ' *'

        per_f1_str = '  '.join(f'{v["per_class_f1"][c]:>6.3f}' for c in CLASSES)
        print(f'{ep:>3}  {tr_loss:>7.4f}  {v["loss"]:>7.4f}  '
              f'{v["macro_f1"]:>8.3f}  {v["acc"]:>6.3f}  {per_f1_str}{star}')

    print(f'\nDone in {(time.time()-t0)/60:.1f} min  |  '
          f'Best macro-F1: {best_f1:.3f}  |  Saved: {ckpt_path}')

    # Plot curves
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle(f'{stage_name} Training', fontsize=13)
    axes[0].plot(history['tr_loss'], label='train')
    axes[0].plot(history['val_loss'], label='val')
    axes[0].set_title('Loss'); axes[0].legend()
    axes[1].plot(history['val_f1'],  label='macro F1')
    axes[1].plot(history['val_acc'], label='accuracy')
    axes[1].set_title('Macro F1 / Accuracy'); axes[1].legend()
    plt.tight_layout()
    curve_path = MODEL_OUT_DIR / f'group_{stage_name}_curves.png'
    plt.savefig(curve_path, dpi=100); plt.close()
    print(f'Curves saved: {curve_path}')

    return model, history


## Run Training

**Stage 1** (web + arctic): runs only if `WEB_DIR` exists and has images.

**Stage 2** (arctic fine-tune): always runs. Uses the Stage 1 checkpoint if available, otherwise trains from ImageNet weights.

In [ ]:
def run_model(model_name):
    """Run full two-stage training for one backbone. Returns final eval results."""
    if model_name == 'efficientnet':
        model = build_efficientnet_b2(num_classes=len(CLASSES)).to(DEVICE)
    else:
        if not INSECTNET_WEIGHTS.exists():
            print(f'ERROR: InsectNet weights not found at {INSECTNET_WEIGHTS}')
            return None
        model = build_insectnet(INSECTNET_WEIGHTS, num_classes=len(CLASSES), unfreeze_last_block=UNFREEZE_LAST_BLOCK).to(DEVICE)

    val_loader         = make_loader(all_samples_for_split, arctic_val_idx,  IMG_SIZE, BATCH,
                                     augment=False, weighted_sampler=False)
    test_loader_arctic = make_loader(arctic_samples,         arctic_test_idx, IMG_SIZE, BATCH,
                                     augment=False, weighted_sampler=False)
    test_loader_web    = make_loader(web_samples,             web_test_idx,   IMG_SIZE, BATCH,
                                     augment=False, weighted_sampler=False)

    # ── Stage 1: web + arctic combined ──────────────────────────────────────
    stage1_ckpt = MODEL_OUT_DIR / f'group_{model_name}_stage1.pth'
    if web_samples:
        combined_samples   = arctic_samples + web_samples
        combined_train_idx = (
            arctic_train_idx +
            [len(arctic_samples) + i for i in web_train_idx]
        )
        combined_counts = {i: counts_arctic[i] + counts_web[i] for i in range(len(CLASSES))}
        train_loader_s1 = make_loader(combined_samples, combined_train_idx,
                                      IMG_SIZE, BATCH, augment=True)
        model, _ = run_stage(
            model, f'{model_name}_stage1_combined', train_loader_s1, val_loader,
            EPOCHS_STAGE1, LR_STAGE1, combined_counts, stage1_ckpt, DEVICE
        )
        print('Stage 1 done. Loading best checkpoint for Stage 2...')
        ckpt = torch.load(stage1_ckpt, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ckpt['state_dict'])
    else:
        print(f'[{model_name}] No web images — skipping Stage 1.')

    # ── Stage 2: arctic fine-tune ────────────────────────────────────────────
    stage2_epochs = EPOCHS_STAGE2_PER_MODEL.get(model_name, EPOCHS_STAGE2)
    stage2_ckpt = MODEL_OUT_DIR / f'group_{model_name}_best.pth'
    if stage2_epochs == 0:
        print(f'[{model_name}] Skipping Stage 2 (stage2_epochs=0). Using Stage 1 checkpoint as final.')
        import shutil
        shutil.copy(stage1_ckpt, stage2_ckpt)
    else:
        train_loader_s2 = make_loader(arctic_samples, arctic_train_idx,
                                       IMG_SIZE, BATCH, augment=True)
        model, _ = run_stage(
            model, f'{model_name}_stage2_arctic', train_loader_s2, val_loader,
            stage2_epochs, LR_STAGE2, counts_arctic, stage2_ckpt, DEVICE
        )

    # ── Final eval ───────────────────────────────────────────────────────────
    ckpt = torch.load(stage2_ckpt, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt['state_dict'])
    criterion_final = weighted_criterion(counts_arctic, CLASSES, DEVICE)
    final      = eval_epoch(model, val_loader,  criterion_final, DEVICE, CLASSES)
    final_test_arctic = eval_epoch(model, test_loader_arctic, criterion_final, DEVICE, CLASSES)
    final_test_web    = eval_epoch(model, test_loader_web,    criterion_final, DEVICE, CLASSES)

    print(f'\n--- {model_name} final report (best checkpoint epoch {ckpt["epoch"]}) ---')
    print(f'Val        MacroF1={final["macro_f1"]:.3f}  Acc={final["acc"]:.3f}')
    print(f'Test Arctic MacroF1={final_test_arctic["macro_f1"]:.3f}  Acc={final_test_arctic["acc"]:.3f}  (real field performance)')
    print(f'Test Web    MacroF1={final_test_web["macro_f1"]:.3f}  Acc={final_test_web["acc"]:.3f}  (generalisation)')
    print('  (Test sets were never seen during training or model selection)')
    if HAS_SKLEARN:
        print('\n-- Val set --')
        print(classification_report(final['labels'], final['preds'],
                                    target_names=CLASSES, digits=3))
        print('\n-- Test Arctic (field crops) --')
        print(classification_report(final_test_arctic['labels'], final_test_arctic['preds'],
                                    target_names=CLASSES, digits=3))
        print('\n-- Test Web (iNaturalist) --')
        print(classification_report(final_test_web['labels'], final_test_web['preds'],
                                    target_names=CLASSES, digits=3))
        cm = confusion_matrix(final['labels'], final['preds'])
        print('Confusion matrix:')
        header = f'{"":20}' + ''.join(f'{c[:8]:>10}' for c in CLASSES)
        print(header)
        for i, cls in enumerate(CLASSES):
            row = f'true {cls[:15]:15}' + ''.join(f'{cm[i,j]:>10}' for j in range(len(CLASSES)))
            print(row)

    # confusion matrix for test arctic
    if HAS_SKLEARN:
        cm_arctic = confusion_matrix(final_test_arctic['labels'], final_test_arctic['preds'])
        print('\nConfusion matrix (Test Arctic):')
        header = f'{"":20}' + ''.join(f'{c[:8]:>10}' for c in CLASSES)
        print(header)
        for i, cls in enumerate(CLASSES):
            row = f'true {cls[:15]:15}' + ''.join(f'{cm_arctic[i,j]:>10}' for j in range(len(CLASSES)))
            print(row)

    (MODEL_OUT_DIR / f'group_{model_name}_results.json').write_text(
        json.dumps({
            'model':              model_name,
            'classes':            CLASSES,
            'best_epoch':         ckpt['epoch'],
            'val_macro_f1':       final['macro_f1'],
            'val_acc':            final['acc'],
            'val_per_class_f1':   final['per_class_f1'],
            'test_arctic_macro_f1':  final_test_arctic['macro_f1'],
            'test_arctic_acc':       final_test_arctic['acc'],
            'test_arctic_per_class': final_test_arctic['per_class_f1'],
            'test_web_macro_f1':     final_test_web['macro_f1'],
            'test_web_acc':          final_test_web['acc'],
            'test_web_per_class':    final_test_web['per_class_f1'],
        }, indent=2))
    return final


# ── Run ───────────────────────────────────────────────────────────────────
all_results = {}

if MODEL in ('efficientnet', 'both'):
    all_results['efficientnet'] = run_model('efficientnet')

if MODEL in ('insectnet', 'both'):
    all_results['insectnet'] = run_model('insectnet')

# ── Comparison summary ────────────────────────────────────────────────────
if MODEL == 'both' and len(all_results) == 2:
    print('\n' + '='*70)
    print('COMPARISON SUMMARY')
    print('='*70)
    header = f'{"Model":15}  {"MacroF1":>8}  {"Acc":>6}  ' + '  '.join(f'{c[:8]:>8}' for c in CLASSES)
    print(header)
    for name, r in all_results.items():
        if r is None:
            continue
        per = '  '.join(f'{r["per_class_f1"][c]:>8.3f}' for c in CLASSES)
        print(f'{name:15}  {r["macro_f1"]:>8.3f}  {r["acc"]:>6.3f}  {per}')
    print('\nPrioritise per-class F1 for rare classes (bumblebee, butterfly_moth, other).')
    print('fly F1 is expected to be highest due to data imbalance.')


## Inference — Predict a Single Crop

In [ ]:
def load_group_classifier(ckpt_path):
    """Load saved group classifier for inference."""
    ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    classes  = ckpt['classes']
    img_size = ckpt['img_size']
    model = torchvision.models.efficientnet_b2(weights=None)
    in_features = model.classifier[-1].in_features
    model.classifier[-1] = nn.Linear(in_features, len(classes))
    model.load_state_dict(ckpt['state_dict'])
    model.eval()
    print(f'Loaded group classifier from {ckpt_path}')
    print(f'  classes: {classes}')
    print(f'  val macro-F1={ckpt["val_macro_f1"]:.3f}')
    return model, img_size, classes


def predict_group(model, img_size, classes, crop_path):
    """Predict pollinator class for a single crop."""
    tf = T.Compose([
        T.Lambda(lambda img: letterbox(img, img_size)),
        T.ToTensor(),
        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    img   = Image.open(crop_path).convert('RGB')
    x     = tf(img).unsqueeze(0)
    with torch.no_grad():
        probs = torch.softmax(model(x), dim=1)[0]
    pred_idx = probs.argmax().item()
    pred     = classes[pred_idx]
    conf     = probs[pred_idx].item()
    print(f'Prediction: {pred}  (confidence={conf:.2%})')
    for i, cls in enumerate(classes):
        print(f'  {cls:20}: {probs[i].item():.2%}')
    return pred, conf


# ── Example usage ─────────────────────────────────────────────────────────
ckpt_path = MODEL_OUT_DIR / 'group_best.pth'
if ckpt_path.exists():
    clf, img_sz, cls_list = load_group_classifier(ckpt_path)
    test_crop = next((ARCTIC_DIR / 'bumblebee').glob('*.jpg'), None)
    if test_crop:
        print(f'\nTesting on: {test_crop.name}')
        predict_group(clf, img_sz, cls_list, test_crop)
    else:
        print('No bumblebee crops found — run annotate.py first')
else:
    print(f'No checkpoint at {ckpt_path} — run training first')
